In [54]:
!pip install openaq requests pandas scikit-learn

In [55]:
import pandas as pd
import numpy as np
import urllib.request
import zipfile

print("Downloading and extracting real-world dataset...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"
zip_path = "AirQualityUCI.zip"
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extract('AirQualityUCI.csv')

df = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')
df = df.dropna(how='all').iloc[:, :15]

# THE FIX: Replace -200 and interpolate realistically instead of flat-lining
df = df.replace(-200, np.nan)
df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')
df = df.set_index('Timestamp')
df = df.drop(columns=['Date', 'Time'])

# Time interpolation draws smooth curves over missing data
df = df.interpolate(method='time').dropna()
df = df.rename(columns={'PT08.S1(CO)': 'Target_Pollutant'})

df.to_csv("ground_aqi_data.csv")
print(f"Cleaned records: {len(df)}")

Cleaned records: 9357


In [56]:
# 1. ADD THIS RIGHT AFTER YOUR CYCLICAL TIME ENCODING CELL
print("Applying Signal Smoothing and removing degraded hardware data...")

# Drop the last 1500 hours (the timeframe where the physical sensors failed)
df = df.iloc[:-1500]

# Apply a 3-hour rolling average to the target to smooth out hardware glitches
# This makes the curve mathematically predictable for the Neural Network
df['Target_Pollutant'] = df['Target_Pollutant'].rolling(window=3, min_periods=1).mean()

print(f"Final Cleaned Dataset Shape: {df.shape}")

Applying Signal Smoothing and removing degraded hardware data...
Final Cleaned Dataset Shape: (7857, 13)


In [57]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [58]:
import pandas as pd
# Read just the first row to check the headers without crashing
temp_df = pd.read_csv("ground_aqi_data.csv", nrows=0)
print("Your actual column names are:")
print(temp_df.columns.tolist())

Your actual column names are:
['Timestamp', 'CO(GT)', 'Target_Pollutant', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


In [59]:
df = pd.read_csv("ground_aqi_data.csv", index_col='Timestamp', parse_dates=True)

# Final safety net to ensure no NaNs crash the neural network
df = df.dropna()

# Find the index of our target variable so the model knows what to predict
target_col_name = 'Target_Pollutant'
target_index = df.columns.get_loc(target_col_name)

# Convert to numpy array and normalize (Crucial for Neural Networks)
raw_data = df.values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(raw_data)
print(f"Data loaded and scaled. Shape: {scaled_data.shape}")

Data loaded and scaled. Shape: (9357, 13)


In [60]:
def create_sequences(data, target_idx, lookback=48, forecast=24):
    X, y = [], []
    for i in range(len(data) - lookback - forecast):
        X.append(data[i : i + lookback])
        # Predicting the target feature for the next 24 hours
        y.append(data[i + lookback : i + lookback + forecast, target_idx])
    return np.array(X), np.array(y)

print("Building sequential memory arrays...")
X, y = create_sequences(scaled_data, target_idx=target_index, lookback=48, forecast=24)

# Sequential Split (No shuffling for time-series forecasting)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True)
print(f"Training sequences: {X_train.shape[0]} | Validation sequences: {X_val.shape[0]}")

Building sequential memory arrays...
Training sequences: 7428 | Validation sequences: 1857


In [61]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

model = Sequential([
    # 1. Convolutional Layer: Extracts dominant features/spikes from the noise
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same',
           input_shape=(X_train.shape[1], X_train.shape[2])),
    MaxPooling1D(pool_size=2),
    BatchNormalization(),

    # 2. Bidirectional LSTM: Reads the time sequence forwards and backwards
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    # 3. Deep Sequential Mapping
    Bidirectional(LSTM(64, return_sequences=False)),
    BatchNormalization(),
    Dropout(0.3),

    # 4. Dense Output Mapping
    Dense(64, activation='relu'),
    Dense(24, activation='linear') # 24 future hours
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [62]:

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# IMPORTANT: Save ONLY the best epoch to prevent overfitting
checkpoint = ModelCheckpoint("aqi_model.keras", monitor='val_loss', save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=26, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)

print("Commencing Deep Learning...")


Commencing Deep Learning...


In [63]:
history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - loss: 0.2520 - mae: 0.3779 - val_loss: 0.0609 - val_mae: 0.1957 - learning_rate: 0.0010
Epoch 2/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0720 - mae: 0.2096 - val_loss: 0.0253 - val_mae: 0.1221 - learning_rate: 0.0010
Epoch 3/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0356 - mae: 0.1469 - val_loss: 0.0216 - val_mae: 0.1212 - learning_rate: 0.0010
Epoch 4/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0272 - mae: 0.1287 - val_loss: 0.0208 - val_mae: 0.1096 - learning_rate: 0.0010
Epoch 5/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.0226 - mae: 0.1170 - val_loss: 0.0157 - val_mae: 0.0992 - learning_rate: 0.0010
Epoch 6/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 0.0199 - mae: 0.1098 - val_loss: 0.0145 - val_mae: 0.0936 - learning_rate: 0.0010
Epoch 7/80
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0185 - mae: 0.1059 - val_loss: 0.0137 - val_mae: 0.0918 - learning_rate: 0.001

In [64]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Raw predictions (still scaled 0-1)
y_pred_scaled = model.predict(X_val)

# 2. Inverse-transform: MinMaxScaler was fit on ALL 13 columns, so to invert
#    only the target column we rebuild dummy rows with the right shape.
def inverse_target(scaled_target_col, scaler, target_idx, n_features):
    dummy = np.zeros((len(scaled_target_col), n_features))
    dummy[:, target_idx] = scaled_target_col
    return scaler.inverse_transform(dummy)[:, target_idx]

n_features = X_train.shape[2]
y_val_flat   = y_val.reshape(-1)
y_pred_flat  = y_pred_scaled.reshape(-1)

y_val_real  = inverse_target(y_val_flat,  scaler, target_index, n_features)
y_pred_real = inverse_target(y_pred_flat, scaler, target_index, n_features)

mae  = mean_absolute_error(y_val_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
r2   = r2_score(y_val_real, y_pred_real)
mape = np.mean(np.abs((y_val_real - y_pred_real) / np.clip(y_val_real, 1e-6, None))) * 100

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

# 3. Accuracy per forecast hour (hour 1 vs hour 24 — accuracy usually drops with horizon)
y_val_2d  = y_val.reshape(-1, 24)
y_pred_2d = y_pred_scaled.reshape(-1, 24)
for h in [0, 5, 11, 17, 23]:
    real_h = inverse_target(y_val_2d[:, h],  scaler, target_index, n_features)
    pred_h = inverse_target(y_pred_2d[:, h], scaler, target_index, n_features)
    print(f"Hour +{h+1:>2}: MAE = {mean_absolute_error(real_h, pred_h):.2f}")

59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
MAE:  56.75
RMSE: 75.13
R²:   0.8838
MAPE: 5.19%
Hour + 1: MAE = 58.44
Hour + 6: MAE = 54.14
Hour +12: MAE = 54.81
Hour +18: MAE = 55.05
Hour +24: MAE = 66.52


In [68]:
import joblib
from google.colab import files

# We use your actual variable name 'scaler' here, but save the file as 'feature_scaler.pkl'
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

# Trigger the downloads
files.download("feature_scaler.pkl")
files.download("target_scaler.pkl")
files.download("aqi_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>